In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructField, StringType, StructType, IntegerType
from datetime import datetime

spark = SparkSession.builder.getOrCreate()

In [21]:
data = [('James', 45, 0, 3), ('Naiara', 36, 0, 1), ('Pedro', 30, 1, 2)]
schema = StructType([
    StructField('nome', StringType(), True),
    StructField('idade', IntegerType(), True),
    StructField('caes', IntegerType(), True),
    StructField('gatos', IntegerType(), True)
])
df = spark.createDataFrame(data, schema)
df.show()

+------+-----+----+-----+
|  nome|idade|caes|gatos|
+------+-----+----+-----+
| James|   45|   0|    3|
|Naiara|   36|   0|    1|
| Pedro|   30|   1|    2|
+------+-----+----+-----+



In [22]:
this_year = datetime.now().year
this_year

2026

In [23]:
df2 = df.withColumn("nascimento", this_year - F.col("idade"))
df2 = df2.withColumn("pets", F.expr("caes + gatos"))
df2 = df2.withColumn("supremacia_gatos", F.expr("gatos - caes"))
df2.show()

+------+-----+----+-----+----------+----+----------------+
|  nome|idade|caes|gatos|nascimento|pets|supremacia_gatos|
+------+-----+----+-----+----------+----+----------------+
| James|   45|   0|    3|      1981|   3|               3|
|Naiara|   36|   0|    1|      1990|   1|               1|
| Pedro|   30|   1|    2|      1996|   3|               1|
+------+-----+----+-----+----------+----+----------------+



In [24]:
df2.explain()

== Physical Plan ==
*(1) Project [nome#102, idade#103, caes#104, gatos#105, (2026 - idade#103) AS nascimento#119, (caes#104 + gatos#105) AS pets#120, (gatos#105 - caes#104) AS supremacia_gatos#121]
+- *(1) Scan ExistingRDD[nome#102,idade#103,caes#104,gatos#105]




In [35]:
df1 = df.select(
    'nome',
    'idade',
    'caes',
    'gatos',
    (this_year - F.col("idade")).alias("nascimento"),
    F.expr("caes + gatos").alias("pets"),
    F.expr("gatos - caes").alias("supremacia_gatos")
)
df1.show()

+------+-----+----+-----+----------+----+----------------+
|  nome|idade|caes|gatos|nascimento|pets|supremacia_gatos|
+------+-----+----+-----+----------+----+----------------+
| James|   45|   0|    3|      1981|   3|               3|
|Naiara|   36|   0|    1|      1990|   1|               1|
| Pedro|   30|   1|    2|      1996|   3|               1|
+------+-----+----+-----+----------+----+----------------+



In [36]:
df1.explain()

== Physical Plan ==
*(1) Project [nome#102, idade#103, caes#104, gatos#105, (2026 - idade#103) AS nascimento#233, (caes#104 + gatos#105) AS pets#234, (gatos#105 - caes#104) AS supremacia_gatos#235]
+- *(1) Scan ExistingRDD[nome#102,idade#103,caes#104,gatos#105]


